<a href="https://colab.research.google.com/github/anisrabiu/PINN-2D-model/blob/main/18b_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#code ref:
#CursorPINN
#a PyTorch PINN code to solve the 2-D Navier–Stokes equations in a greenhouse of size 6.5 m × 4.5 m.
#
#24/09/2025

#Improve

In [ ]:
import os
import math
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from typing import Tuple, Dict


In [ ]:
# ---------------------------
# Configuration
# ---------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32

torch.set_default_dtype(DTYPE)

# Geometry
W = 6.5
H = 4.5
xc = 3.25
R = 3.565
yc = 0.435
# ridge parameters
sv = 0.56  # half ridge width (m) as provided
apex_x = xc
apex_y = 4.5
roof_vent_half_width = 0.45  # 0.9 m total

# Physics
rho = 1.225  # kg/m^3
nu = 1.5e-5  # m^2/s (kinematic viscosity of air)

# Time
T_final = 24 * 60 * 60.0  # one day, seconds

# Network/training
HID = 64
LAYERS = 6
LR = 1e-3
EPOCHS = 1000
SAVE_EVERY = 100
BATCH_F_INNER = 10000
BATCH_BC = 2000
BATCH_INIT = 4000
BATCH_DATA = 256

# Use a hardcoded path or relative path instead of __file__
OUTPUT_DIR = './outputs_gothic_ns' # Example: using a relative path
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Update the CSV path based on the file listing
CSV_PATH = '/content/wind velocity 28022023.csv'


# Add characteristic scales for normalization
Lc = W
Hc = H
Tc = 24 * 3600.0
Uc = 0.8  # characteristic velocity scale (m/s) used to non-dimensionalize outputs

In [ ]:
# ---------------------------
# Utility: roof curve and helpers
# ---------------------------

def roof_arc_y(x: torch.Tensor) -> torch.Tensor:
	"""Circular arc y for |x-xc| >= sv region.
	Pass torch tensor x.
	"""
	dx = x - xc
	# ensure inside circle domain
	val = yc + torch.sqrt(torch.clamp(R ** 2 - dx ** 2, min=0.0))
	return val


def ridge_base_height() -> float:
	x_left = xc - sv
	return float(roof_arc_y(torch.tensor([x_left]))[0].item())


RIDGE_BASE_Y = ridge_base_height()


def roof_y(x: torch.Tensor) -> torch.Tensor:
	"""Piecewise roof profile: arc for |x-xc|>=sv, straight lines to apex otherwise."""
	x = x.clone()
	dy = torch.zeros_like(x)
	mask_arc_left = (x <= xc - sv)
	mask_arc_right = (x >= xc + sv)
	mask_ridge = (~mask_arc_left) & (~mask_arc_right)
	# arc regions
	dy[mask_arc_left] = roof_arc_y(x[mask_arc_left])
	dy[mask_arc_right] = roof_arc_y(x[mask_arc_right])
	# ridge lines
	if mask_ridge.any():
		# two straight lines meeting at apex
		x_left = xc - sv
		y_left = RIDGE_BASE_Y
		x_right = xc + sv
		y_right = RIDGE_BASE_Y
		# left facet line eq through (x_left,y_left) and (apex_x,apex_y)
		m_left = (apex_y - y_left) / (apex_x - x_left)
		b_left = y_left - m_left * x_left
		# right facet
		m_right = (apex_y - y_right) / (apex_x - x_right)
		b_right = y_right - m_right * x_right
		left_sel = mask_ridge & (x <= xc)
		right_sel = mask_ridge & (x > xc)
		dy[left_sel] = m_left * x[left_sel] + b_left
		dy[right_sel] = m_right * x[right_sel] + b_right
	return dy


def inside_domain(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
	"""Return boolean mask: points below roof and within rectangle bounds."""
	return (x >= 0.0) & (x <= W) & (y >= 0.0) & (y <= roof_y(x))



In [ ]:

# ---------------------------
# Vent schedule and inflow
# ---------------------------

def vents_open(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Return 1 if time is between 08:00 and 18:00, else 0. Accept torch tensor."""
	start = 8 * 3600.0
	end = 18 * 3600.0
	return ((t_seconds >= start) & (t_seconds <= end)).to(t_seconds.dtype)


def inflow_speed(t_seconds: torch.Tensor) -> torch.Tensor:
	"""Inflow magnitude in m/s when vents open. Bumped to 0.8 m/s for clearer signal."""
	return 0.8 * vents_open(t_seconds)


In [ ]:

# ---------------------------
# Data loading (optional)
# ---------------------------

def load_probe_csv(csv_path: str) -> pd.DataFrame:
	if not os.path.exists(csv_path):
		return pd.DataFrame(columns=['x', 'y', 't', 'S1'])
	df = pd.read_csv(csv_path)
	# ensure numeric
	df['t'] = pd.to_numeric(df['t'], errors='coerce')
	df['S1'] = pd.to_numeric(df['S1'], errors='coerce')
	df = df.dropna()
	return df


PROBE_DF = load_probe_csv(CSV_PATH)
PROBE_X = 3.25
PROBE_Y = 1.2


In [ ]:
# ---------------------------
# Network
# ---------------------------
class MLP(nn.Module):
	def __init__(self, in_dim: int, out_dim: int, hidden: int, layers: int):
		super().__init__()
		mods = []
		mods.append(nn.Linear(in_dim, hidden))
		mods.append(nn.Tanh())
		for _ in range(layers - 2):
			mods.append(nn.Linear(hidden, hidden))
			mods.append(nn.Tanh())
		mods.append(nn.Linear(hidden, out_dim))
		self.net = nn.Sequential(*mods)

	def forward(self, x):
		return self.net(x)


# Model maps (x_hat,y_hat,t_hat) -> (u_hat,v_hat,p_hat)
model = MLP(3, 3, HID, LAYERS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


In [ ]:
# ---------------------------
# Autograd helpers
# ---------------------------

def gradients(y, x):
	return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True, retain_graph=True, only_inputs=True)[0]



In [ ]:

# ---------------------------
# Samplers
# ---------------------------

def sample_interior(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	# rejection sampling under roof
	xs = torch.rand(n * 2, dtype=DTYPE) * W
	ys = torch.rand(n * 2, dtype=DTYPE) * H
	mask = inside_domain(xs, ys)
	xs = xs[mask][:n]
	ys = ys[mask][:n]
	if xs.shape[0] < n:
		# pad by recursion
		x2, y2, _ = sample_interior(n - xs.shape[0])
		xs = torch.cat([xs, x2])
		ys = torch.cat([ys, y2])
	ts = torch.rand(n, dtype=DTYPE) * T_final
	return xs, ys, ts


def sample_ground(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	x = torch.rand(n, dtype=DTYPE) * W
	y = torch.zeros(n, dtype=DTYPE)
	t = torch.rand(n, dtype=DTYPE) * T_final
	return x, y, t


def sample_side(n: int, side: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
	"""Sample points on left or right wall, label vent mask for open segment y in [0.3,1.2].
	Bias 70% of samples to the vent span to strengthen BCs."""
	k_vent = int(0.7 * n)
	k_rest = n - k_vent
	if side == 'left':
		xv = torch.zeros(k_vent, dtype=DTYPE)
		xr = torch.zeros(k_rest, dtype=DTYPE)
	elif side == 'right':
		xv = torch.full((k_vent,), W, dtype=DTYPE)
		xr = torch.full((k_rest,), W, dtype=DTYPE)
	else:
		raise ValueError('side must be left or right')
	yv = 0.3 + torch.rand(k_vent, dtype=DTYPE) * (1.2 - 0.3)
	yr = torch.rand(k_rest, dtype=DTYPE) * 1.9
	x = torch.cat([xv, xr], dim=0)
	y = torch.cat([yv, yr], dim=0)
	t = torch.rand(n, dtype=DTYPE) * T_final
	vent_mask = (y >= 0.3) & (y <= 1.2)
	return x, y, t, vent_mask.to(torch.uint8)


def sample_roof(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
	"""Sample roof boundary points and label those on roof vent segment."""
	x = torch.rand(n, dtype=DTYPE) * W
	y = roof_y(x)
	t = torch.rand(n, dtype=DTYPE) * T_final
	# mark roof-vent points near apex along x within width
	vent_mask = (x >= (xc - roof_vent_half_width)) & (x <= (xc + roof_vent_half_width))
	return x, y, t, vent_mask.to(torch.uint8)


def sample_initial(n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	x, y, _ = sample_interior(n)
	t = torch.zeros_like(x)
	return x, y, t


def sample_probe_from_csv(df: pd.DataFrame, max_n: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
	if df is None or df.empty:
		return torch.empty(0), torch.empty(0), torch.empty(0), torch.empty(0)
	df2 = df.copy()
	# Use up to max_n rows sampled uniformly
	if len(df2) > max_n:
		df2 = df2.sample(n=max_n, random_state=np.random.randint(0, 1_000_000))
	x = torch.full((len(df2),), PROBE_X, dtype=DTYPE)
	y = torch.full((len(df2),), PROBE_Y, dtype=DTYPE)
	t = torch.tensor(df2['t'].values, dtype=DTYPE)
	spd = torch.tensor(df2['S1'].values, dtype=DTYPE)
	return x, y, t, spd

	return torch.empty(0), torch.empty(0), torch.empty(0), torch.empty(0)


In [ ]:
# ---------------------------
# Physics residuals
# ---------------------------

def pde_residuals(x, y, t):
	# Normalize inputs
	xh = x / Lc
	yh = y / Hc
	th = t / Tc
	inp = torch.stack([xh, yh, th], dim=1).to(DEVICE)
	inp.requires_grad_(True)
	# Network outputs (dimensionless)
	out = model(inp)
	u_hat = out[:, 0:1]
	v_hat = out[:, 1:2]
	p_hat = out[:, 2:3]
	# De-scale to physical variables for PDE
	u = Uc * u_hat
	v = Uc * v_hat
	p = rho * (Uc ** 2) * p_hat
	# chain-rule factors
	dx = 1.0 / Lc
	dy = 1.0 / Hc
	dt = 1.0 / Tc
	# gradients w.r.t normalized inputs
	u_xh = gradients(u, inp)[:, 0:1]
	u_yh = gradients(u, inp)[:, 1:2]
	u_th = gradients(u, inp)[:, 2:3]
	v_xh = gradients(v, inp)[:, 0:1]
	v_yh = gradients(v, inp)[:, 1:2]
	v_th = gradients(v, inp)[:, 2:3]
	p_xh = gradients(p, inp)[:, 0:1]
	p_yh = gradients(p, inp)[:, 1:2]
	# convert to physical derivatives
	u_x = dx * u_xh
	u_y = dy * u_yh
	u_t = dt * u_th
	v_x = dx * v_xh
	v_y = dy * v_yh
	v_t = dt * v_th
	p_x = dx * p_xh
	p_y = dy * p_yh
	# Laplacians via second derivatives on normalized inputs
	u_xxh = gradients(u_xh, inp)[:, 0:1]
	u_yyh = gradients(u_yh, inp)[:, 1:2]
	v_xxh = gradients(v_xh, inp)[:, 0:1]
	v_yyh = gradients(v_yh, inp)[:, 1:2]
	u_xx = (dx ** 2) * u_xxh
	u_yy = (dy ** 2) * u_yyh
	v_xx = (dx ** 2) * v_xxh
	v_yy = (dy ** 2) * v_yyh
	# residuals
	r_u = u_t + (u * u_x + v * u_y) + (1.0 / rho) * p_x - nu * (u_xx + u_yy)
	r_v = v_t + (u * v_x + v * v_y) + (1.0 / rho) * p_y - nu * (v_xx + v_yy)
	r_c = u_x + v_y
	return r_u, r_v, r_c


In [ ]:

# ---------------------------
# Loss terms
# ---------------------------

def loss_pde(n_inner: int):
	x, y, t = sample_interior(n_inner)
	x = x.to(DEVICE); y = y.to(DEVICE); t = t.to(DEVICE)
	r_u, r_v, r_c = pde_residuals(x, y, t)
	return (r_u ** 2).mean() + (r_v ** 2).mean() + (r_c ** 2).mean()


def loss_walls_and_vents(n_bc: int):
	loss = 0.0
	# ground (no-slip)
	xg, yg, tg = sample_ground(n_bc)
	inp_g = torch.stack([xg / Lc, yg / Hc, tg / Tc], dim=1).to(DEVICE)
	u_hat_g, v_hat_g, _ = model(inp_g).split(1, dim=1)
	ug = Uc * u_hat_g
	vg = Uc * v_hat_g
	loss += (ug ** 2 + vg ** 2).mean()
	# left and right walls
	for side, sign in [('left', +1.0), ('right', -1.0)]:
		xw, yw, tw, vent_mask = sample_side(n_bc, side)
		inp_w = torch.stack([xw / Lc, yw / Hc, tw / Tc], dim=1).to(DEVICE)
		u_hat, v_hat, _ = model(inp_w).split(1, dim=1)
		u = Uc * u_hat
		v = Uc * v_hat
		open_mask = vents_open(tw.to(DEVICE)) * vent_mask.to(torch.float32).to(DEVICE)
		u_in = inflow_speed(tw.to(DEVICE)) * sign
		loss_in = ((u - u_in.view(-1, 1)) ** 2 + v ** 2) * open_mask.view(-1, 1)
		closed_mask = 1.0 - open_mask
		loss_wall = (u ** 2 + v ** 2) * closed_mask.view(-1, 1)
		loss += (loss_in.mean() + loss_wall.mean())
	# roof
	xr, yr, tr, vent_mask_r = sample_roof(n_bc)
	inp_r = torch.stack([xr / Lc, yr / Hc, tr / Tc], dim=1).to(DEVICE)
	u_hat, v_hat, p_hat = model(inp_r).split(1, dim=1)
	u = Uc * u_hat
	v = Uc * v_hat
	p = rho * (Uc ** 2) * p_hat
	open_mask_r = vents_open(tr.to(DEVICE)) * vent_mask_r.to(torch.float32).to(DEVICE)
	loss_roof_vent = (p ** 2) * open_mask_r.view(-1, 1)
	loss_roof_wall = (u ** 2 + v ** 2) * (1.0 - open_mask_r).view(-1, 1)
	loss += loss_roof_vent.mean() + loss_roof_wall.mean()
	# pressure reference at a single interior point
	x_ref = torch.tensor([xc], dtype=DTYPE)
	y_ref = torch.tensor([1.5], dtype=DTYPE)
	t_ref = torch.tensor([12 * 3600.0], dtype=DTYPE)
	inp_ref = torch.stack([x_ref / Lc, y_ref / Hc, t_ref / Tc], dim=1).to(DEVICE)
	p_ref = rho * (Uc ** 2) * model(inp_ref)[:, 2:3]
	loss += 0.1 * (p_ref ** 2).mean()
	return loss


def loss_initial(n_init: int):
	x, y, t = sample_initial(n_init)
	inp = torch.stack([x / Lc, y / Hc, t / Tc], dim=1).to(DEVICE)
	u_hat, v_hat, _ = model(inp).split(1, dim=1)
	u = Uc * u_hat
	v = Uc * v_hat
	return (u ** 2 + v ** 2).mean()


def loss_data(max_n: int):
	if PROBE_DF is None or PROBE_DF.empty:
		return torch.tensor(0.0, device=DEVICE)
	x, y, t, spd = sample_probe_from_csv(PROBE_DF, max_n)
	if x.numel() == 0:
		return torch.tensor(0.0, device=DEVICE)
	inp = torch.stack([x / Lc, y / Hc, t / Tc], dim=1).to(DEVICE)
	u_hat, v_hat, _ = model(inp).split(1, dim=1)
	pred_spd = torch.sqrt((Uc * u_hat) ** 2 + (Uc * v_hat) ** 2)
	return ((pred_spd.view(-1) - spd.to(DEVICE)) ** 2).mean()


# Weights for combined loss
W_PDE = 1.0
W_BC = 10.0  # strengthen boundary conditions
W_INIT = 1.0
W_DATA = 0.1  # reduce pull to mostly-zero probe data


In [ ]:

# ---------------------------
# Visualization helpers
# ---------------------------

def outline_roof(ax):
	xs = np.linspace(0, W, 400, dtype=np.float64)
	xs_t = torch.tensor(xs, dtype=DTYPE)
	ys = roof_y(xs_t).detach().cpu().numpy()
	ax.plot(xs, ys, 'k-', lw=1.5)
	ax.plot([0, 0], [0, 1.9], 'k--', lw=1)
	ax.plot([W, W], [0, 1.9], 'k--', lw=1)
	# side vents
	ax.plot([0, 0], [0.3, 1.2], 'b', lw=4, alpha=0.5)
	ax.plot([W, W], [0.3, 1.2], 'b', lw=4, alpha=0.5)
	# roof vent span
	ax.axvspan(xc - roof_vent_half_width, xc + roof_vent_half_width, color='b', alpha=0.15)


def predict_field(t_seconds: float, nx: int = 120, ny: int = 80):
	xs = np.linspace(0, W, nx)
	ys = np.linspace(0, H, ny)
	X, Y = np.meshgrid(xs, ys)
	xv = torch.tensor(X.flatten(), dtype=DTYPE)
	yv = torch.tensor(Y.flatten(), dtype=DTYPE)
	mask = inside_domain(xv, yv)
	inp = torch.stack([xv[mask] / Lc, yv[mask] / Hc, torch.full_like(xv[mask], t_seconds) / Tc], dim=1).to(DEVICE)
	with torch.no_grad():
		uvp = model(inp)
	u = torch.zeros_like(xv)
	v = torch.zeros_like(yv)
	u[mask] = (Uc * uvp[:, 0])
	v[mask] = (Uc * uvp[:, 1])
	speed = torch.sqrt(u ** 2 + v ** 2)
	return X, Y, u.view(Y.shape).cpu().numpy(), v.view(Y.shape).cpu().numpy(), speed.view(Y.shape).cpu().numpy()


def plot_field(t_seconds: float, fname: str):
	fig, ax = plt.subplots(1, 1, figsize=(7, 4))
	X, Y, U, V, S = predict_field(t_seconds)
	c = ax.contourf(X, Y, S, levels=30, cmap='viridis')
	ax.quiver(X[::3, ::3], Y[::3, ::3], U[::3, ::3], V[::3, ::3], color='w', scale=20)
	outline_roof(ax)
	ax.set_aspect('equal')
	ax.set_xlim(0, W)
	ax.set_ylim(0, H)
	ax.set_title(f'|u| at t={t_seconds/3600:.2f} h')
	fig.colorbar(c, ax=ax, label='m/s')
	fig.tight_layout()
	fig.savefig(fname, dpi=150)
	plt.close(fig)


def plot_probe_timeseries(fname: str):
	times = PROBE_DF['t'].values if not PROBE_DF.empty else np.arange(0, T_final + 1, 600)
	x = torch.full((len(times),), PROBE_X, dtype=DTYPE)
	y = torch.full((len(times),), PROBE_Y, dtype=DTYPE)
	t = torch.tensor(times, dtype=DTYPE)
	inp = torch.stack([x / Lc, y / Hc, t / Tc], dim=1).to(DEVICE)
	with torch.no_grad():
		uvp = model(inp)
	pred = torch.sqrt((Uc * uvp[:, 0]) ** 2 + (Uc * uvp[:, 1]) ** 2).cpu().numpy()
	fig, ax = plt.subplots(1, 1, figsize=(7, 3))
	ax.plot(times / 3600.0, pred, label='PINN')
	if not PROBE_DF.empty:
		ax.plot(PROBE_DF['t'].values / 3600.0, PROBE_DF['S1'].values, 'k--', lw=1, alpha=0.7, label='Measured S1')
	ax.set_xlabel('time (h)')
	ax.set_ylabel('|u| at probe (m/s)')
	ax.legend()
	ax.grid(True, ls=':')
	fig.tight_layout()
	fig.savefig(fname, dpi=150)
	plt.close(fig)


In [ ]:
# ---------------------------
# Training loop
# ---------------------------

def train():
	start_time = time.time()
	for epoch in range(1, EPOCHS + 1):
		optimizer.zero_grad()
		Lpde = loss_pde(BATCH_F_INNER)
		Lbc = loss_walls_and_vents(BATCH_BC)
		Linit = loss_initial(BATCH_INIT)
		Ldata = loss_data(BATCH_DATA)
		total = W_PDE * Lpde + W_BC * Lbc + W_INIT * Linit + W_DATA * Ldata
		total.backward()
		optimizer.step()

		if epoch % 100 == 0 or epoch == 1:
			print(f"Epoch {epoch:5d} | L: {total.item():.4e} | PDE: {Lpde.item():.4e} | BC: {Lbc.item():.4e} | IC: {Linit.item():.4e} | DATA: {Ldata.item():.4e}")

		if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
			# save checkpoint
			ckpt_path = os.path.join(OUTPUT_DIR, f'ckpt_{epoch}.pt')
			torch.save({'model': model.state_dict(), 'epoch': epoch}, ckpt_path)
			# plots
			plot_probe_timeseries(os.path.join(OUTPUT_DIR, f'probe_{epoch}.png'))
			plot_field(12 * 3600.0, os.path.join(OUTPUT_DIR, f'field_12h_{epoch}.png'))
			plot_field(2 * 3600.0, os.path.join(OUTPUT_DIR, f'field_02h_{epoch}.png'))
			# also dump a small json with losses
			with open(os.path.join(OUTPUT_DIR, f'loss_{epoch}.json'), 'w') as f:
				json.dump({'epoch': epoch, 'Lpde': float(Lpde.item()), 'Lbc': float(Lbc.item()), 'Linit': float(Linit.item()), 'Ldata': float(Ldata.item())}, f, indent=2)

	print(f'Training finished in {(time.time()-start_time)/60:.1f} min')


if __name__ == '__main__':
	train()


Epoch     1 | L: 4.8003e+00 | PDE: 6.1634e-06 | BC: 4.7743e-01 | IC: 9.4832e-03 | DATA: 1.6467e-01
Epoch   100 | L: 3.2377e+00 | PDE: 3.5282e-03 | BC: 3.2080e-01 | IC: 9.0778e-03 | DATA: 1.7113e-01
Epoch   200 | L: 2.2399e+00 | PDE: 7.1781e-03 | BC: 2.2043e-01 | IC: 1.1142e-02 | DATA: 1.7271e-01
Epoch   300 | L: 1.7482e+00 | PDE: 6.4420e-03 | BC: 1.7203e-01 | IC: 4.6064e-03 | DATA: 1.6887e-01
Epoch   400 | L: 8.5226e-01 | PDE: 7.9251e-03 | BC: 8.2168e-02 | IC: 5.7859e-03 | DATA: 1.6866e-01
Epoch   500 | L: 6.6815e-01 | PDE: 7.1725e-03 | BC: 6.4333e-02 | IC: 1.3685e-03 | DATA: 1.6276e-01
Epoch   600 | L: 6.0582e-01 | PDE: 8.2436e-03 | BC: 5.7926e-02 | IC: 2.0818e-03 | DATA: 1.6239e-01
Epoch   700 | L: 4.6669e-01 | PDE: 9.8226e-03 | BC: 4.3956e-02 | IC: 7.6238e-04 | DATA: 1.6540e-01
Epoch   800 | L: 3.5074e-01 | PDE: 1.3096e-02 | BC: 3.2019e-02 | IC: 8.8003e-04 | DATA: 1.6576e-01
Epoch   900 | L: 2.8103e-01 | PDE: 1.7398e-02 | BC: 2.4624e-02 | IC: 9.6735e-04 | DATA: 1.6424e-01
Epoch  100